# Концентрация электронов по связанной плазмон-фононной моде L+

Модель та же, что в ваших статьях: ω±² = ½[(ωLO² + ωp²) ± √((ωLO² + ωp²)² − 4ωp²ωTO²)],
ωp² = n·e² / (ε0·ε∞·m*). Нажмите **Run All**.

Файлы `../data/raw/own/*.txt` появляются после `scripts/import_own.py`.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from ramancore.io import read_spectrum
from ramancore.peaks import curves_on_grid
from ramancore.semiconductors import LOPC_PARAMS, coupled_modes, lplus_concentration

LOPC_PARAMS["GaAs"]   # параметры по умолчанию; свои можно передать: TO=..., LO=...

## 1. Один спектр

In [ ]:
my_file = "../data/raw/own/own_gaas_ar154e_294K.txt"  # ← подставьте свой файл
x, y, meta = read_spectrum(my_file)
r = lplus_concentration(x, y, material="GaAs")
print(f"L+ = {r['omega']:.1f} ± {r['omega_err']:.1f} см⁻¹,  n = {r['n']:.2e} см⁻³")
for w in r["warnings"]:
    print("•", w)

In [ ]:
fig = go.Figure()
fig.add_scatter(x=x, y=y, name="спектр", line=dict(color="lightgray"))
if r["fit"] is not None:
    xf = np.linspace(r["fit"]["x"][0], r["fit"]["x"][-1], 500)
    base, comps, total = curves_on_grid(r["table"], r["fit"], xf)
    fig.add_scatter(x=xf, y=total, name="подгонка L+ (по сглаженному)")
fig.update_layout(xaxis_title="Рамановский сдвиг, см⁻¹", yaxis_title="Интенсивность",
                  xaxis_range=[150, 900])
fig.show()

## 2. Сравнение с Холлом

`hall` — концентрация по Холлу из таблицы образцов. Для p-типа и малых n
L+ не видна (её место занято LO) — это ожидаемо.

In [ ]:
samples = {  # файл: концентрация по Холлу, см⁻³
    "own_gaas_ar154s": 7.11e17,
    "own_gaas_ar97": 1.47e18,
    "own_gaas_ar154e_294K": 1.89e18,
}
rows = []
for name, hall in samples.items():
    xs, ys, _ = read_spectrum(f"../data/raw/own/{name}.txt")
    res = lplus_concentration(xs, ys)
    rows.append({"образец": name, "Холл": hall, "L+, см⁻¹": res["omega"],
                 "n по Раману": res["n"], "Раман/Холл": res["n"] / hall})
pd.DataFrame(rows).round(2)

## 3. Расчётные кривые L−, L+, ωp от концентрации

In [ ]:
n = np.logspace(16, 19, 300)
lm, lp, wp = coupled_modes(n, "GaAs")
fig = go.Figure()
for curve, label in [(lm, "L−"), (lp, "L+"), (wp, "ωp")]:
    fig.add_scatter(x=n, y=curve, name=label)
for row in rows:
    fig.add_scatter(x=[row["Холл"]], y=[row["L+, см⁻¹"]], mode="markers",
                    name=row["образец"], marker=dict(size=10))
fig.update_layout(xaxis_type="log", xaxis_title="n, см⁻³", yaxis_title="см⁻¹",
                  yaxis_range=[0, 800])
fig.show()